In [1]:
from math import atan2, cos, sin, sqrt, pi, degrees
from pathlib import Path
import os
import subprocess
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from astropy import units as u
from astropy.coordinates import get_body_barycentric_posvel, solar_system_ephemeris
from astropy.time import Time

from scipy.integrate import solve_ivp
from scipy.optimize import least_squares



In [ ]:
ASTEPH_DIR = '/home/pavel/Programs/astEph' # локальная программа astEph с MPCORB.DAT
ASTEPH_RATE_STEP = 300.0    # с - шаг численного дифференцирования для скоростей из astEph
ASTID = '24445'
MU = 2.9591220828411956E-04 # а.е.^3/сут^2 - GM Солнца (как _GMS в astEph, = k^2)
C_AU_DAY = 299792.458*86400/1.495978707E+08 # скорость света, а.е./сут
EPS_J2000 = np.radians(23.43929111)         # наклон эклиптики J2000 (как в astEph)


In [3]:
def analyze_vector_D(pos, n):
    # Вычисляем JD0
    jd0 = pos['JD'].mean()

    # Инициализируем массивы
    D = np.zeros(3)
    dotD = np.zeros(3)
    ddotD = np.zeros(3)

    # Вычисляем компоненты D
    pos['D0'] = np.cos(np.radians(pos['RA'])) * np.cos(np.radians(pos['DEC']))
    pos['D1'] = np.sin(np.radians(pos['RA'])) * np.cos(np.radians(pos['DEC']))
    pos['D2'] = np.sin(np.radians(pos['DEC']))

    # Инициализируем результаты
    results = []

    for j in range(3):
        DD = pos[f'D{j}']
        JD = pos['JD']

        # Вычисляем x = JD - JD0
        x = JD - jd0

        # Построение модели D = sum C_k * x^k
        X = np.column_stack([x**k for k in range(n+1)])

        # Метод наименьших квадратов
        C, residuals, rank, singular_values = np.linalg.lstsq(X, DD, rcond=None)

        D[j] = C[0]

        # Вычисляем первые и вторые производные в точке x=0
        if n >= 1:
            dotD[j] = C[1]
        else:
            dotD[j] = 0
        if n >= 2:
            ddotD[j] = 2 * C[2]
        else:
            ddotD[j] = 0
            
    return jd0, D, dotD, ddotD

In [4]:
RADEC_CSV = Path("24445_20260929_radec.csv")

pos = pd.read_csv(RADEC_CSV)
n = 3  # порядок полиномиальной аппроксимации
JD0, D, dotD, ddotD = analyze_vector_D(pos, n)
JD0, D, dotD, ddotD

(np.float64(2461313.1702109934),
 array([ 0.43689826, -0.87564611,  0.20582467]),
 array([ 0.0019453 , -0.00098478, -0.00831885]),
 array([0.0024996 , 0.00153091, 0.0008566 ]))

In [5]:
# Результаты аппроксимации, вычисленные в предыдущей ячейке.
JD0, D, dotD, ddotD

(np.float64(2461313.1702109934),
 array([ 0.43689826, -0.87564611,  0.20582467]),
 array([ 0.0019453 , -0.00098478, -0.00831885]),
 array([0.0024996 , 0.00153091, 0.0008566 ]))

In [6]:
def _run_asteph(target, jd, location):
    """Один вызов astEph: (RA, DEC в градусах, V) на момент jd (UTC) для обсерватории location."""
    exe = os.path.join(ASTEPH_DIR, 'astEph')
    out = subprocess.run([exe, str(target), f'{jd:.8f}', f'obscode={location}'], cwd=ASTEPH_DIR,
                         capture_output=True, text=True, timeout=120)
    if out.returncode != 0:
        raise RuntimeError(f'astEph: {out.stderr.strip() or out.stdout.strip()}')
    val = {l.split('=')[0].strip(): l.split('=', 1)[1].split()[0] for l in out.stdout.splitlines() if '=' in l}
    return np.degrees(float(val['RA'])), np.degrees(float(val['DEC'])), float(val['Vmag'])

def get_asteph_ephemeris(target, JD, location):
    """Та же эфемерида, что get_horizons_ephemeris, но из локальной программы astEph (ASTEPH_DIR).
    astEph дает только положение и V, поэтому скорости считаются по положениям в моменты
    jd -+ ASTEPH_RATE_STEP (с), а DELTA не заполняется. Первый вызов делается отдельно: при новой
    эпохе MPCORB.DAT astEph сначала считает и сохраняет свой кэш интегрирования (~10 с). Остальные
    вызовы идут параллельно."""
    JD = np.atleast_1d(np.asarray(JD, dtype=float))
    if not os.path.isfile(os.path.join(ASTEPH_DIR, 'astEph')):
        raise FileNotFoundError(f'нет программы {ASTEPH_DIR}/astEph (сборка: cc -O2 -o astEph astEph.c -lm)')
    h = ASTEPH_RATE_STEP/86400
    jds = np.concatenate([JD, JD-h, JD+h])
    first = _run_asteph(target, jds[0], location)
    with ThreadPoolExecutor(max_workers=8) as ex:
        rest = list(ex.map(lambda jd: _run_asteph(target, jd, location), jds[1:]))
    res = np.array([first] + rest).reshape(3, len(JD), 3)
    (ra, dec, v), (ra1, dec1, _), (ra2, dec2, _) = [res[k].T for k in range(3)]
    dra = ((ra2-ra1+180) % 360 - 180)*np.cos(np.radians(dec))*3600/(2*ASTEPH_RATE_STEP/3600)
    ddec = (dec2-dec1)*3600/(2*ASTEPH_RATE_STEP/3600)
    return pd.DataFrame(dict(JD=JD, RA_EPH=ra, DEC_EPH=dec, dRAcosD=dra, dDEC=ddec, # arcsec/hour
                             VMAG=v, DELTA=np.nan))


def get_observer_state(JD, location='217', asteph_dir=ASTEPH_DIR):
    """Гелиоцентрические положение R (а.е.), скорость V (а.е./сут) и ускорение A (а.е./сут^2)
    наблюдателя в обсерватории с кодом MPC location на момент JD (UTC), экваториальная система J2000.
    Считает программа astEph (режим --observer). Возвращает три numpy-массива по 3 компоненты."""
    out = subprocess.run([os.path.join(asteph_dir, 'astEph'), '--observer', f'{float(JD):.8f}',
                          f'obscode={location}'], cwd=asteph_dir, capture_output=True, text=True, timeout=300)
    if out.returncode != 0:
        raise RuntimeError(f'astEph: {out.stderr.strip() or out.stdout.strip()}')
    val = {k.strip(): v for k, v in (l.split('=', 1) for l in out.stdout.splitlines() if '=' in l)}
    R, V, A = (np.array(val[k].split(), dtype=float) for k in ('R', 'V', 'A'))
    return R, V, A

In [7]:
g, dot_g, ddot_g = get_observer_state(JD0, location='217', asteph_dir=ASTEPH_DIR)
g, dot_g, ddot_g

(array([0.99593577, 0.09864487, 0.04278753]),
 array([-0.00201476,  0.01579488,  0.00677486]),
 array([-1.33101313e-03,  6.40727226e-04, -9.24954877e-06]))

In [8]:
def laplace_coefficients(D, dotD, ddotD, g, ddot_g):
    """Коэффициенты уравнений C d = C2 + C3 r^-3 и r^2 = d^2 + 2 C1 d + C0."""
    DxdD = np.cross(D, dotD)
    return dict(C=ddotD @ DxdD, C0=g @ g, C1=g @ D, C2=-ddot_g @ DxdD, C3=-MU*(g @ DxdD))

coef = laplace_coefficients(D, dotD, ddotD, g, ddot_g)
pd.Series(coef)

NameError: name 'MU' is not defined

In [ ]:
D_MIN = 1e-4     # а.е. (~15000 км) - решения ближе к наблюдателю считаются ложными
ROOT_INDEX = 0   # какое из оставшихся решений брать, если их несколько

def solve_rd(C, C0, C1, C2, C3):
    """Все положительные вещественные корни уравнения Лагранжа и соответствующие d (из lagrange_rd.tex)."""
    al, be = C2 / C, C3 / C
    a = -(al**2 + 2*C1*al + C0)
    b = -2*be*(al + C1)
    c = -be**2
    roots = np.roots([1, 0, a, 0, 0, b, 0, 0, c]) # коэффициенты при r^8, r^7, ..., r^0
    sols = []
    for r in roots:
        if abs(r.imag) < 1e-10 and r.real > 0:
            r = r.real
            sols.append((r, al + be / r**3))
    return sorted(sols)

sols = solve_rd(**coef)
for r_, d_ in sols:
    print(f'r = {r_:.6f} а.е., d = {d_:+.6f} а.е.' + ('' if d_ > D_MIN else '  <- отброшено (d <= D_MIN)'))
good = [(r_, d_) for r_, d_ in sols if d_ > D_MIN]
if not good:
    raise RuntimeError('нет физического решения (d > D_MIN)')
r, d = good[min(ROOT_INDEX, len(good)-1)]
print(f'выбрано: r = {r:.6f} а.е., d = {d:.6f} а.е.')